In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from prophet import Prophet

# Loading Dataset

In [5]:
df = pd.read_csv(r'C:\Aryan Pashte\Final Placement\Python placement\Python projects\Superstore.csv', encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [7]:
print(df.shape)

(9994, 21)


In [8]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

# Checking for null values

In [9]:
print(df.isnull().sum())

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


# Checking for duplicates

In [10]:
print(df.duplicated().sum())

0


# Converting Date columns

In [11]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

df['Ship Date'] = pd.to_datetime(df['Ship Date'])

In [12]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


# Feature Engineerig

In [13]:
df['Year'] = df['Order Date'].dt.year

In [14]:
df['Month'] = df['Order Date'].dt.month_name()

In [15]:
df['Month_No'] = df['Order Date'].dt.month

In [16]:
df['Quarter'] = df['Order Date'].dt.quarter

In [17]:
df['Shipping Days'] = (
    df['Ship Date']
    -
    df['Order Date']
).dt.days

In [18]:
df['Profit Margin %'] = (
    df['Profit']
    /
    df['Sales']
)*100

In [19]:
df['Discount Amount'] = (
    df['Sales']
    *
    df['Discount']
)

In [20]:
df['Loss Order'] = np.where(
    df['Profit'] < 0,
    'Loss',
    'Profit'
)

# Customer Intelligence

In [21]:
customer_clv = (
    df.groupby(
        ['Customer ID','Customer Name']
    )
    .agg({
        'Sales':'sum',
        'Profit':'sum',
        'Order ID':'nunique'
    })
    .reset_index()
)

In [22]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Discount,Profit,Year,Month,Month_No,Quarter,Shipping Days,Profit Margin %,Discount Amount,Loss Order
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,November,11,4,3,16.00,0.000000,Profit
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,November,11,4,3,30.00,0.000000,Profit
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,June,6,2,4,47.00,0.000000,Profit
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,October,10,4,7,-40.00,430.909875,Loss
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,October,10,4,7,11.25,4.473600,Profit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,0.20,4.1028,2014,January,1,1,2,16.25,5.049600,Profit
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.00,15.6332,2017,February,2,1,5,17.00,0.000000,Profit
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.20,19.3932,2017,February,2,1,5,7.50,51.715200,Profit
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.00,13.3200,2017,February,2,1,5,45.00,0.000000,Profit


In [23]:
customer_clv.columns = [
    'Customer ID',
    'Customer Name',
    'Lifetime Sales',
    'Lifetime Profit',
    'Total Orders'
]

In [24]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Discount,Profit,Year,Month,Month_No,Quarter,Shipping Days,Profit Margin %,Discount Amount,Loss Order
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,41.9136,2016,November,11,4,3,16.00,0.000000,Profit
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,0.00,219.5820,2016,November,11,4,3,30.00,0.000000,Profit
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,0.00,6.8714,2016,June,6,2,4,47.00,0.000000,Profit
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.45,-383.0310,2015,October,10,4,7,-40.00,430.909875,Loss
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,0.20,2.5164,2015,October,10,4,7,11.25,4.473600,Profit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,0.20,4.1028,2014,January,1,1,2,16.25,5.049600,Profit
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.00,15.6332,2017,February,2,1,5,17.00,0.000000,Profit
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.20,19.3932,2017,February,2,1,5,7.50,51.715200,Profit
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,0.00,13.3200,2017,February,2,1,5,45.00,0.000000,Profit


# Top Customers

In [25]:
top_customers = customer_clv.sort_values(
    'Lifetime Sales',
    ascending=False
).head(20)

# Product Intelligence

In [27]:
product_analysis = (
    df.groupby(
        'Product Name'
    )
    .agg({
        'Sales':'sum',
        'Profit':'sum',
        'Quantity':'sum'
    })
    .reset_index()
)

# Top Products

In [28]:
top_products = (
    product_analysis
    .sort_values(
        'Sales',
        ascending=False
    )
    .head(20)
)

# Most Profitable Products

In [29]:
top_profit_products = (
    product_analysis
    .sort_values(
        'Profit',
        ascending=False
    )
    .head(20)
)

# Category Analysis

In [30]:
category_analysis = (
    df.groupby(
        'Category'
    )
    .agg({
        'Sales':'sum',
        'Profit':'sum',
        'Discount':'mean'
    })
    .reset_index()
)

# Region Analysis

In [31]:
region_analysis = (
    df.groupby(
        'Region'
    )
    .agg({
        'Sales':'sum',
        'Profit':'sum'
    })
    .reset_index()
)

# Monthly Sales

In [34]:
monthly_sales = (
    df.groupby(
        pd.Grouper(
            key='Order Date',
            freq='M'
        )
    )['Sales']
    .sum()
    .reset_index()
)

C:\Users\praka\AppData\Local\Temp\ipykernel_31968\2689195386.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pd.Grouper(


# Sales Forecasting

In [35]:
forecast_df = monthly_sales.copy()

forecast_df.columns = [
    'ds',
    'y'
]

# Training Model

In [36]:
model = Prophet()

model.fit(forecast_df)

18:47:45 - cmdstanpy - INFO - Chain [1] start processing
18:47:46 - cmdstanpy - INFO - Chain [1] done processing


In [37]:
future = model.make_future_dataframe(
    periods=6,
    freq='M'
)

C:\pythonProject\venv\Lib\site-packages\prophet\forecaster.py:1875: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


In [38]:
forecast = model.predict(
    future
)

In [39]:
forecast_final = forecast[
    [
        'ds',
        'yhat',
        'yhat_lower',
        'yhat_upper'
    ]
]

In [40]:
forecast_final.to_csv(
    "Sales_Forecast.csv",
    index=False
)

# Exporting Cleaned Data

In [41]:
df.to_csv(
    "Retail_Sales_Cleaned.csv",
    index=False
)

customer_clv.to_csv(
    "Customer_Analysis.csv",
    index=False
)

product_analysis.to_csv(
    "Product_Analysis.csv",
    index=False
)